# KAN-83 E05 smoothing experiment report

## 1. Summary

The 28-day climatology blend had the lowest dev +24 h MAE (3.791), but its paired interval crosses zero. No candidate beats the smoother after the 26-comparison Bonferroni sensitivity.


## 2. Hypothesis and decision rule

The preregistrations hypothesize improved +24 h MAE and compare dev MAE and paired CIs. Protocol section 6 names the 72-hour CI as primary, but the individual preregistrations did not name a block length; all 72-hour readings are exploratory. No holdout or promotion decision is made here.


## 3. Setup

Frozen snapshot `a00cfc9edd4c3fb5f7b946e8871be9e5a88fe2bb8ef56b91d895101da41c0208`; five weekly dev folds starting 2026-08-31 through 2026-09-28 UTC; expanding training, weekly refit at fold start minus 24 hours. Hourly observations assume availability at timestamp + 1 hour. Features use only timestamp <= origin - 1h and availability <= origin.

## 4. EDA and data checks

Five stations, 4,200 common +24 h pairs and 100% baseline-eligible coverage in each reported run. Air4Thai-era and KAN-74 cutoff exclusions are enforced by the shared runner. There are 15 observed spike pairs, so spike precision and recall are undefined under the 50-spike rule. Leakage, stale readings and smoother equivalence are covered by targeted tests.

## 5. Results

Lowest dev MAE per family: MA24 3.926, EWMA 6 h 3.874, beta 0.7 3.829, 28-day climatology/persistence blend 3.791. All CIs cross zero. Beta 0.6 has an unadjusted 72 h CI below zero, but is fragile, and no candidate beats the smoother after 26-comparison correction. The common fixed-smoother reference is E04 run `da9ad074c138430199b42f1c2ef23d21`. Full rows and run IDs are in `docs/experiments/e05-smoothing.md`.


## 6. Statistical tests and diagnostics

Beta 0.6 unadjusted 72 h CI [-0.066, -0.007]; 24 h [-0.070, 0.002]; 168 h [-0.051, -0.007]: fragile. DM p=0.008 (167 h), sensitivity p=0.084 (47 h) and 0.112 (23 h). Protocol section 8 requires no adjustment for descriptive dev exploration; its holdout correction applies to up to two finalists. An additional post-hoc 26-comparison Bonferroni sensitivity gives beta 0.6 a 99.8077% 72 h CI [-0.081, 0.009], tracked as `e3fb0fa2e6dc4d4e9050a71208fa56ae`. It was the only 95% CI excluding zero, so none excludes zero after correction. There are 27 original-grid runs for 26 candidates plus the diagnostic run; MA3 was rerun to optimize execution, with identical scores and counters.


## 7. Figures

Lowest dev MAE per family and the E04 smoother reference. No candidate beats the smoother after 26-comparison correction.


In [ ]:
import matplotlib.pyplot as plt
import mlflow

from pm25_forecast.experiments.tracking import tracking_uri

mlflow.set_tracking_uri(tracking_uri())
runs = {
    "MA 24h": "3a4d8f968d104fe4818703cc29b28adb",
    "EWMA 6h": "d2e50e44ba8343598ab50e5e75fcada4",
    "Beta 0.7": "3f8a628c51014da4ac62ffbbe58245ee",
    "Climatology blend 28d": "444eecf8cc5d4dc492e157ee0fa51104",
    "Smoother": "da9ad074c138430199b42f1c2ef23d21",
}
values = [mlflow.get_run(run).data.metrics["mae_24h"] for run in runs.values()]
plt.barh(list(runs), values)
plt.xlabel("+24 h MAE (µg/m³)")
plt.title("Best dev variant per family")
plt.show()

## 8. Conclusion, limitations, next

No candidate beats the smoother after 26-comparison correction. Do not promote from these exploratory results. The lowest MAE is uncertain; beta 0.6 is sensitive to block size and falls short of the minimum-effect threshold. Results cover one season and assumed availability, not observed arrival time. A later preregistered final-selection process must decide without holdout tuning.
